# Bài 21: Luồng quang học và Ước lượng chuyển động (Optical Flow and Motion Estimation)

Luồng quang học (optical flow) ước lượng chuyển động biểu kiến giữa hai khung hình của một video: với mỗi điểm ảnh (hoặc một tập điểm được chọn), một vector 2D $(u, v)$ mô tả nó đã di chuyển tới đâu. Bài này suy ra phương pháp **Lucas-Kanade** kinh điển từ chính tensor cấu trúc đã dùng để phát hiện góc ở Bài 20, đối mặt với **bài toán khẩu độ (aperture problem)** cơ bản, đối chiếu nó với phương pháp **Horn-Schunck** tối ưu toàn cục, và kết thúc bằng luồng dày đặc (dense flow) với phương pháp **Farnebäck**.

In [ ]:
# Download the sample images used in this lesson, if not already present
import os, urllib.request

for _f in ['army10.png', 'army11.png', 'building.png']:
    _path = f"../img/{_f}"
    if not os.path.exists(_path):
        os.makedirs("../img", exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/sbirchfield/sbirchfield.github.io/main/cvintro/img/{_f}", _path)

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Giả định độ sáng không đổi (The brightness constancy assumption)
Luồng quang học giả định cường độ của một điểm không thay đổi khi nó di chuyển: $I(x, y, t) = I(x+u, y+v, t+1)$. Khai triển Taylor bậc nhất vế phải cho ra **phương trình ràng buộc luồng quang học (optical flow constraint equation)**:

$$I_x u + I_y v + I_t = 0$$

trong đó $I_x, I_y$ là các gradient không gian (Bài 12) và $I_t$ là hiệu cường độ giữa hai khung hình. Đây là **một phương trình với hai ẩn số** (chuyển động $u$ và $v$) tại mỗi điểm ảnh &mdash; tự nó không đủ thông tin để giải ra luồng.

## Bài toán khẩu độ (The aperture problem)
Nhìn qua một cửa sổ nhỏ vào một biên đang di chuyển, chỉ có chuyển động **vuông góc với biên** là nhìn thấy được; chuyển động **dọc theo biên** hoàn toàn không tạo ra thay đổi nào thấy được, nên phép đo cục bộ không thể nhận ra nó. Đây chính là lý do phương trình ràng buộc luồng bị thiếu ràng buộc: nó chỉ ràng buộc thành phần của $(u,v)$ dọc theo hướng gradient $(I_x, I_y)$, để thành phần vuông góc hoàn toàn không bị ràng buộc bởi phương trình đó.

In [ ]:
size = 160
rect_topleft = (50, 60)
rect_w, rect_h = 60, 40

def make_rect_frame(dx=0, dy=0, value=200):
    img = np.zeros((size, size), dtype=np.uint8)
    x0, y0 = rect_topleft[0] + dx, rect_topleft[1] + dy
    cv2.rectangle(img, (x0, y0), (x0 + rect_w, y0 + rect_h), value, -1)
    return img

true_motion = (8, 6)           # right and down
vertical_only_motion = (0, 6)  # same downward shift, but no horizontal component

frame_true = make_rect_frame(*true_motion)
frame_vertical = make_rect_frame(*vertical_only_motion)

# A small aperture window straddling the rectangle's TOP (horizontal) edge, centered well
# away from any corner and comfortably inside the canvas, so there are no border effects.
win_half = 10
wx, wy = rect_topleft[0] + rect_w // 2, rect_topleft[1]

def with_window(img_gray, color=(255, 0, 0)):
    vis = cv2.cvtColor(img_gray, cv2.COLOR_GRAY2RGB)
    cv2.rectangle(vis, (wx - win_half, wy - win_half), (wx + win_half, wy + win_half), color, 1)
    return vis

fig, axes = plt.subplots(1, 2, figsize=(7, 4))
axes[0].imshow(with_window(frame_true))
axes[0].set_title(f'True motion (u,v)={true_motion}\n(right + down)', fontsize=9)
axes[1].imshow(with_window(frame_vertical))
axes[1].set_title(f'Vertical-only motion (u,v)={vertical_only_motion}\n(down only)', fontsize=9)
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

window_true = frame_true[wy - win_half:wy + win_half, wx - win_half:wx + win_half]
window_vertical = frame_vertical[wy - win_half:wy + win_half, wx - win_half:wx + win_half]
print('inside the red window, the two (different!) motions look pixel-for-pixel identical:',
      np.array_equal(window_true, window_vertical))

Hai khung hình thực ra thể hiện hai chuyển động toàn cục khác nhau &mdash; một cái di chuyển sang phải và xuống dưới, cái kia chỉ xuống dưới &mdash; vậy mà bên trong cửa sổ đỏ (nằm vắt qua cạnh trên của hình chữ nhật, một biên hoàn toàn nằm ngang) chúng không thể phân biệt được. Thành phần ngang nằm *dọc theo* biên đó, nên nó không để lại dấu vết cục bộ nào; chỉ có thành phần dọc chung, *vuông góc* với biên, là nhìn thấy được. Một phép đo cục bộ tại cửa sổ này thực sự không thể phân biệt hai chuyển động này.

Ảnh động này cho thấy cạnh của hình chữ nhật thực sự di chuyển theo đường chéo — cả thành phần dọc-theo-biên lẫn thành phần vuông góc đều khác không và thay đổi cùng nhau. Cửa sổ màu cam được cố định trên biên; nội dung bị cắt của nó (bên phải) chỉ dịch chuyển theo chiều dọc, không bao giờ theo chiều ngang, bất kể mũi tên chuyển động thực (bên trái) hướng về đâu. Bài toán khẩu độ, nhìn thấy trực tiếp thay vì chỉ được khẳng định.

<img src="../animations/aperture_problem_demo.gif" alt="Ảnh động bài toán khẩu độ: cạnh của hình chữ nhật di chuyển theo đường chéo trong khi nội dung bị cắt của cửa sổ khẩu độ chỉ dịch chuyển theo chiều dọc" style="display: block; margin: 0 auto; max-width: 680px;">

## Lucas-Kanade: giải bài toán khẩu độ bằng một cửa sổ (Lucas-Kanade: solving the aperture problem with a window)
<a href="../references.html#lucas-kanade-1981">Cách khắc phục của Lucas và Kanade (1981)</a>: giả định luồng $(u,v)$ là **không đổi trên một cửa sổ nhỏ**, rồi kết hợp phương trình ràng buộc luồng từ mọi điểm ảnh trong cửa sổ đó thành một hệ bình phương tối thiểu thừa ràng buộc (overdetermined):

$$\underbrace{\begin{bmatrix}\sum I_x^2 & \sum I_xI_y \\ \sum I_xI_y & \sum I_y^2\end{bmatrix}}_{M}\begin{bmatrix}u\\v\end{bmatrix} = -\begin{bmatrix}\sum I_xI_t\\ \sum I_yI_t\end{bmatrix}$$

$M$ chính xác là tensor cấu trúc ở Bài 20! Đây không phải là trùng hợp: giải hệ này đòi hỏi $M$ khả nghịch, tức là có hai giá trị riêng lớn &mdash; chính là điều kiện "đặc trưng tốt để theo dõi" của Shi-Tomasi. Một vùng phẳng ($M$ gần bằng 0) hoặc một biên (một giá trị riêng nhỏ &mdash; lại là bài toán khẩu độ) cho ra một hệ điều kiện kém hoặc suy biến; một góc cho ra hệ điều kiện tốt. **Các góc theo dõi được vì đúng cùng lý do khiến chúng là góc tốt.**

In [ ]:
rng = np.random.default_rng(0)
frame1 = np.zeros((200, 200), dtype=np.uint8)
for _ in range(20):
    x, y = rng.integers(20, 180, 2)
    radius = rng.integers(5, 15)
    cv2.circle(frame1, (x, y), radius, int(rng.integers(100, 255)), -1)

small_motion = (0.6, 0.4)  # sub-pixel motion, well inside the linear (Taylor) approximation's validity
shift = np.float32([[1, 0, small_motion[0]], [0, 1, small_motion[1]]])
frame2_small = cv2.warpAffine(frame1, shift, (200, 200))

Ix = cv2.Sobel(frame1.astype(np.float64), cv2.CV_64F, 1, 0, ksize=3, scale=1 / 8)
Iy = cv2.Sobel(frame1.astype(np.float64), cv2.CV_64F, 0, 1, ksize=3, scale=1 / 8)
It = frame2_small.astype(np.float64) - frame1.astype(np.float64)


def lucas_kanade_at(x, y, half_win=7):
    x, y = int(round(x)), int(round(y))
    ix = Ix[y - half_win:y + half_win + 1, x - half_win:x + half_win + 1].ravel()
    iy = Iy[y - half_win:y + half_win + 1, x - half_win:x + half_win + 1].ravel()
    it = It[y - half_win:y + half_win + 1, x - half_win:x + half_win + 1].ravel()
    A = np.stack([ix, iy], axis=1)
    solution, *_ = np.linalg.lstsq(A, -it, rcond=None)
    return solution

corners = cv2.goodFeaturesToTrack(frame1, maxCorners=30, qualityLevel=0.1, minDistance=10)
flows = np.array([lucas_kanade_at(p[0][0], p[0][1]) for p in corners])

print(f'true motion:        {small_motion}')
print(f'manual LK estimate: ({flows[:, 0].mean():.3f}, {flows[:, 1].mean():.3f})  (averaged over {len(corners)} corners)')

### Trực quan hóa ảnh, các đặc trưng được theo dõi và chuyển động của chúng (Visualizing the image, the tracked features, and their motion)

In [ ]:
arrow_scale = 15  # true motion here is sub-pixel, so exaggerate the arrows to make them visible

vis_small = cv2.cvtColor(frame1, cv2.COLOR_GRAY2RGB)
for (x, y), (u, v) in zip(corners[:, 0], flows):
    p0 = (int(round(x)), int(round(y)))
    p1 = (int(round(x + u * arrow_scale)), int(round(y + v * arrow_scale)))
    cv2.arrowedLine(vis_small, p0, p1, (255, 0, 0), 1, tipLength=0.3)
    cv2.circle(vis_small, p0, 2, (0, 255, 0), -1)

plt.imshow(vis_small)
plt.title(f'Corners (green) and estimated motion (red, {arrow_scale}x exaggerated)\ntrue motion = {small_motion}')
plt.axis('off')
plt.show()

### Chuyển động lớn hơn cần lặp (Larger motions need iteration)
Phép giải tuyến tính một lần này dựa vào xấp xỉ Taylor, vốn chỉ đúng với chuyển động nhỏ. Với độ dịch chuyển lớn hơn, cùng cách tiếp cận một lần đó sẽ suy giảm:

In [ ]:
large_motion = (4.0, 3.0)
shift_large = np.float32([[1, 0, large_motion[0]], [0, 1, large_motion[1]]])
frame2_large = cv2.warpAffine(frame1, shift_large, (200, 200))

It_large = frame2_large.astype(np.float64) - frame1.astype(np.float64)

def lucas_kanade_large(x, y, half_win=7):
    x, y = int(round(x)), int(round(y))
    ix = Ix[y - half_win:y + half_win + 1, x - half_win:x + half_win + 1].ravel()
    iy = Iy[y - half_win:y + half_win + 1, x - half_win:x + half_win + 1].ravel()
    it = It_large[y - half_win:y + half_win + 1, x - half_win:x + half_win + 1].ravel()
    A = np.stack([ix, iy], axis=1)
    solution, *_ = np.linalg.lstsq(A, -it, rcond=None)
    return solution

flows_large_manual = np.array([lucas_kanade_large(p[0][0], p[0][1]) for p in corners])

next_pts, status, _ = cv2.calcOpticalFlowPyrLK(frame1, frame2_large, corners, None, winSize=(15, 15), maxLevel=2)
cv_flow = (next_pts - corners).reshape(-1, 2)[status.ravel() == 1]

print(f'true motion:                          {large_motion}')
print(f'single-shot manual LK estimate:       ({flows_large_manual[:, 0].mean():.3f}, {flows_large_manual[:, 1].mean():.3f})  <- degraded')
print(f'cv2.calcOpticalFlowPyrLK estimate:    ({cv_flow[:, 0].mean():.3f}, {cv_flow[:, 1].mean():.3f})  <- accurate')

`calcOpticalFlowPyrLK` của OpenCV xử lý chuyển động lớn bằng cách chạy Lucas-Kanade **lặp đi lặp lại** (biến dạng lại và tuyến tính hóa lại cho đến khi hội tụ) trên một **kim tự tháp ảnh** (Bài 11) &mdash; ước lượng thô trên một phiên bản nhỏ, mờ của ảnh trước, rồi tinh chỉnh từng tầng. Sự kết hợp này cho phép nó khôi phục chính xác các chuyển động lớn dù phép xấp xỉ tuyến tính bên dưới chỉ đúng cục bộ, mỗi lần một bước nhỏ.

### Trực quan hóa các vector luồng thưa (Visualizing sparse flow vectors)
Mỗi góc được theo dõi được vẽ thành một chấm xanh lục tại vị trí của nó trong khung hình đầu tiên, với một mũi tên đỏ chỉ tới nơi Lucas-Kanade tìm thấy nó trong khung hình thứ hai (mũi tên được phóng đại 3 lần vì chuyển động thực ở đây chỉ vài điểm ảnh).

In [ ]:
arrow_scale = 3  # exaggerate the arrows a bit so they're easier to see

vis = cv2.cvtColor(frame1, cv2.COLOR_GRAY2RGB)
for (p0,), (p1,), ok in zip(corners, next_pts, status.ravel()):
    if not ok:
        continue
    p0 = p0.astype(int)
    p1_exaggerated = np.round(p0 + (p1 - p0) * arrow_scale).astype(int)
    cv2.arrowedLine(vis, tuple(p0), tuple(p1_exaggerated), (255, 0, 0), 1, tipLength=0.3)
    cv2.circle(vis, tuple(p0), 2, (0, 255, 0), -1)

plt.imshow(vis)
plt.title(f'Tracked corners, true motion = {large_motion} (arrows {arrow_scale}x exaggerated)')
plt.axis('off')
plt.show()

### Theo dõi điểm qua một đoạn video ngắn (Tracking points through a short video)
Để thấy Lucas-Kanade chạy qua nhiều hơn hai khung hình, ta tạo một chuỗi tổng hợp ngắn bằng cách liên tục biến dạng một bức ảnh tĩnh duy nhất với một phép biến đổi affine thay đổi mượt mà (một chút lia, xoay và phóng to mỗi khung hình) &mdash; mô phỏng một camera đang di chuyển. Các góc Shi-Tomasi được phát hiện một lần trên khung hình 0, rồi được theo dõi từ khung hình này sang khung hình khác bằng `cv2.calcOpticalFlowPyrLK`, nối đầu ra của mỗi khung hình thành đầu vào của khung hình tiếp theo.

In [ ]:
building = cv2.imread('../img/building.png', cv2.IMREAD_GRAYSCALE)
h, w = building.shape
center = (w / 2, h / 2)

# A short synthetic "video": warp the one real photo with a smoothly changing
# affine transform per frame, simulating a panning, slightly zooming camera.
n_frames = 14
video = []
for t in range(n_frames):
    M = cv2.getRotationMatrix2D(center, angle=1.1 * t, scale=1.0 - 0.006 * t)
    M[0, 2] += 1.8 * t
    M[1, 2] += 1.0 * t
    video.append(cv2.warpAffine(building, M, (w, h), borderMode=cv2.BORDER_REPLICATE))

# Detect once on frame 0, then chain cv2.calcOpticalFlowPyrLK frame-to-frame.
pts = cv2.goodFeaturesToTrack(video[0], maxCorners=25, qualityLevel=0.1, minDistance=12)
lk_params = dict(winSize=(15, 15), maxLevel=3,
                  criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01))

prev_pts = pts
for t in range(1, n_frames):
    prev_pts, status, _ = cv2.calcOpticalFlowPyrLK(video[t - 1], video[t], prev_pts, None, **lk_params)

print(f'{len(pts)} corners detected on frame 0, {int(status.sum())} still tracked by frame {n_frames}')

Ảnh động này phát chuỗi trên từng khung hình, vẽ quỹ đạo tích lũy của mỗi điểm được theo dõi trong quá trình chạy &mdash; cùng lệnh gọi `cv2.calcOpticalFlowPyrLK` ở trên, chạy trên một bức ảnh thật thay vì một mẫu thử tổng hợp. Các điểm trên góc mạnh (khung cửa sổ, đường mái nhà) tồn tại suốt cả chuỗi; một vài điểm trôi sang các kết cấu yếu hoặc lặp lại và bị loại bỏ khi bộ theo dõi mất độ tin cậy.

<img src="../animations/lucas_kanade_tracking_demo.gif" alt="Ảnh động theo dõi điểm Lucas-Kanade trên một chuỗi lia camera tổng hợp ngắn từ một bức ảnh thật, cho thấy quỹ đạo các góc được theo dõi tích lũy qua 14 khung hình" style="display: block; margin: 0 auto; max-width: 500px;">

## Horn-Schunck: một phương án toàn cục thay cho luồng theo cửa sổ (Horn-Schunck: a global alternative to windowed flow)
Cửa sổ của Lucas-Kanade là một giả định trơn *cục bộ*: luồng không đổi trên một vùng lân cận nhỏ, được ước lượng độc lập tại mỗi điểm. Cùng năm đó, <a href="../references.html#horn-schunck-1981">Horn và Schunck (1981)</a> đề xuất một phương án *toàn cục*: thay vì nhiều phép giải độc lập trên từng cửa sổ, cực tiểu hóa một hàm năng lượng duy nhất trên toàn ảnh cùng lúc, đánh đổi giữa mức độ luồng thỏa mãn phương trình ràng buộc luồng quang học và mức độ luồng thay đổi mượt mà giữa các điểm ảnh lân cận:

$$E(u,v) = \sum_{x,y} \underbrace{(I_x u + I_y v + I_t)^2}_{\text{data term}} \;+\; \alpha^2 \underbrace{\left(\|\nabla u\|^2 + \|\nabla v\|^2\right)}_{\text{smoothness term}}$$

Số hạng trơn (smoothness term) là thứ làm cho phương pháp này mang tính toàn cục: nó ràng buộc luồng của mỗi điểm ảnh với luồng của các lân cận, nên việc cực tiểu hóa $E$ (ví dụ bằng các cập nhật lặp Gauss-Seidel) cho phép các ước lượng luồng đáng tin cậy gần biên *lan truyền* vào các vùng phẳng, không có kết cấu, vốn không có thông tin cục bộ nào của riêng chúng &mdash; chính là kiểu thất bại mà bài toán khẩu độ gây ra ở mức tồi tệ nhất.

In [ ]:
hs_img1 = np.zeros((120, 120), dtype=np.uint8)
cv2.rectangle(hs_img1, (40, 40), (80, 80), 200, -1)  # one textured square, otherwise flat
hs_true_motion = (3.0, 2.0)
hs_shift = np.float32([[1, 0, hs_true_motion[0]], [0, 1, hs_true_motion[1]]])
hs_img2 = cv2.warpAffine(hs_img1, hs_shift, (120, 120))

def horn_schunck(I1, I2, alpha=5.0, n_iters=100):
    Ix = cv2.Sobel(I1.astype(np.float64), cv2.CV_64F, 1, 0, ksize=3, scale=1 / 8)
    Iy = cv2.Sobel(I1.astype(np.float64), cv2.CV_64F, 0, 1, ksize=3, scale=1 / 8)
    It = I2.astype(np.float64) - I1.astype(np.float64)
    u, v = np.zeros_like(Ix), np.zeros_like(Ix)
    neighbor_avg = np.array([[0, 1, 0], [1, 0, 1], [0, 1, 0]], dtype=np.float64) / 4.0
    for _ in range(n_iters):
        u_avg, v_avg = cv2.filter2D(u, -1, neighbor_avg), cv2.filter2D(v, -1, neighbor_avg)
        correction = (Ix * u_avg + Iy * v_avg + It) / (alpha ** 2 + Ix ** 2 + Iy ** 2)
        u, v = u_avg - Ix * correction, v_avg - Iy * correction
    return u, v

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
for ax, n_iters in zip(axes, [1, 20, 200]):
    u, v = horn_schunck(hs_img1, hs_img2, n_iters=n_iters)
    magnitude = np.hypot(u, v)
    im = ax.imshow(magnitude, cmap='viridis', vmin=0, vmax=np.hypot(*hs_true_motion))
    ax.set_title(f'{n_iters} iterations', fontsize=9)
    ax.axis('off')
plt.suptitle(f'Flow magnitude spreading outward from the square (true motion = {hs_true_motion})')
plt.tight_layout()
plt.show()

Chỉ sau 1 vòng lặp, luồng chỉ khác không ngay tại các cạnh của hình vuông, nơi có thông tin gradient cục bộ; đến 200 vòng lặp, vầng sáng đó đã lan rộng ra khỏi các cạnh vào vùng phẳng xung quanh, dù vẫn còn xa mới đến được mọi điểm ảnh &mdash; một bộ giải thực tế sẽ chạy nhiều vòng lặp hơn nữa (hoặc dùng kim tự tháp, cùng mẹo mà Lucas-Kanade dùng cho chuyển động lớn) để hội tụ ở mọi nơi. Ngay cả sự lan truyền một phần này đã cho thấy cơ chế đang hoạt động: tính trơn lan truyền thông tin vào các vùng không có số hạng dữ liệu của riêng chúng.

## Phương pháp Farnebäck cho luồng quang học dày đặc (Farnebäck's method for dense optical flow)
Phép tối ưu toàn cục của Horn-Schunck thanh lịch nhưng tốn kém và hội tụ chậm; với luồng dày đặc (mọi điểm ảnh), <a href="../references.html#farneback-2003">phương pháp Farnebäck (2003)</a> &mdash; được cung cấp dưới dạng `cv2.calcOpticalFlowFarneback` &mdash; là một phương pháp mới hơn (nhưng vẫn mang tính cổ điển). Nó vẫn cục bộ, giống Lucas-Kanade, nhưng bỏ giả định luồng-không-đổi-trong-cửa-sổ, thay vào đó xấp xỉ cục bộ mỗi vùng lân cận bằng một đa thức và so sánh các khai triển đa thức giữa các khung hình.

Để trực quan hóa một trường luồng dày đặc, vector $(u, v)$ của mỗi điểm ảnh được mã hóa màu thành một điểm ảnh HSV: **sắc độ mã hóa hướng** (góc quanh bánh xe màu) và **độ sáng (giá trị) mã hóa tốc độ** (độ lớn) &mdash; chuyển động nhanh thì sáng, chuyển động chậm thì tối, và mỗi hướng cụ thể (trái, lên-phải, v.v.) ánh xạ tới một màu cụ thể. Trên một video thật có các vật thể chuyển động độc lập, cách mã hóa màu này ngay lập tức tách các chuyển động khác nhau chỉ trong một cái nhìn &mdash; chính vì vậy nó là cách chuẩn để trực quan hóa trường luồng dày đặc.

<img src="../figures/flow_colorwheel.png" alt="Chú giải bánh xe màu: sắc độ mã hóa hướng (phải=đỏ, xuống=vàng-lục, trái=lục lam, lên=lam-tím), độ sáng mã hóa tốc độ, tâm tối nghĩa là chậm" style="display: block; margin: 0 auto; max-width: 320px;">

In [ ]:
def flow_to_color(flow):
    magnitude, angle = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    hsv = np.zeros(flow.shape[:2] + (3,), dtype=np.uint8)
    hsv[..., 0] = angle * 180 / np.pi / 2       # hue = direction
    hsv[..., 1] = 255                            # full saturation
    hsv[..., 2] = cv2.normalize(magnitude, None, 0, 255, cv2.NORM_MINMAX)  # value = speed
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)

### Một video thật: chuỗi Army (A real video: the Army sequence)
Bây giờ hãy xét một cặp khung hình từ bộ đánh giá chuẩn (benchmark) Middlebury, hiển thị bên dưới. Luân phiên giữa hai khung hình giúp dễ dàng nhìn thấy trực tiếp chuyển động &mdash; hãy quan sát những người lính đồ chơi, quả bóng và nền dịch chuyển khác nhau thế nào.

<img src="../img/army_flicker.gif" alt="Chuỗi Army hai khung hình nhấp nháy từ Middlebury" style="display: block; margin: 0 auto; max-width: 500px;">

Tính luồng quang học trên cặp khung hình này bằng Farnebäck và &mdash; để so sánh &mdash; một phương pháp dựa trên học sâu xếp hạng hàng đầu.

In [ ]:
im1 = cv2.imread('../img/army10.png', cv2.IMREAD_GRAYSCALE)
im2 = cv2.imread('../img/army11.png', cv2.IMREAD_GRAYSCALE)
imflow = cv2.calcOpticalFlowFarneback(im1, im2, None, pyr_scale=0.5, levels=3,
                                     winsize=30, iterations=3, poly_n=5, poly_sigma=1.2, flags=0)

army_sota = cv2.cvtColor(cv2.imread('../img/army_sota_flow.jpg'), cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.8))
axes[0].imshow(im1, cmap='gray')
axes[0].set_title('Frame 1')
axes[1].imshow(flow_to_color(imflow))
axes[1].set_title('Farnebäck (classical)')
axes[2].imshow(army_sota)
axes[2].set_title('Top Middlebury benchmark method')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

Dù Farnebäck tách được những người lính, quả bóng và nền thành các màu riêng biệt, các ranh giới của nó mềm và nhòe qua các cạnh của vật thể. Phương pháp dựa trên học sâu tạo ra các ranh giới sắc nét, sạch sẽ hơn nhiều quanh mỗi người lính và quả bóng, với ít hiện tượng lem sang nền hơn hẳn. Khác biệt lớn về chất lượng giữa các đầu ra này giúp minh họa tại sao học sâu lại phổ biến đến vậy trong thị giác máy tính.

<p class="image-attribution">Nguồn ảnh: bộ đánh giá <a href="https://vision.middlebury.edu/flow/" target="_blank" rel="noopener">Middlebury Optical Flow</a>; kết quả luồng chuẩn từ bảng xếp hạng <a href="https://vision.middlebury.edu/flow/eval/results/results-e1.php" target="_blank" rel="noopener">Middlebury flow evaluation</a>.</p>

### Bài tập (Exercises)
1. Chạy lại phép so sánh Lucas-Kanade thưa với `large_motion = (10.0, 8.0)`. `cv2.calcOpticalFlowPyrLK` có còn khôi phục chính xác không? Đến mức nào (hãy thử các chuyển động ngày càng lớn) thì nó bắt đầu thất bại, và tại sao bạn kỳ vọng kim tự tháp giúp đẩy giới hạn đó ra xa hơn?
2. Sửa cảnh tổng hợp để các hình tròn di chuyển với vận tốc *khác nhau* (ví dụ một nửa di chuyển theo một hướng, một nửa theo hướng khác). Chạy lại phần trực quan hóa màu Farnebäck dày đặc và xác nhận hai chuyển động hiển thị thành hai màu riêng biệt.
3. Trong ví dụ bài toán khẩu độ, hãy dựng một mảng nhỏ có kết cấu (không phải biên) thay vì một biên thẳng, và chỉ ra rằng *cả hai* thành phần của một vector chuyển động tùy ý đều khôi phục được từ nó &mdash; khác với trường hợp biên, ở đây không nên có hướng chuyển động nào là vô hình.